# 📓 Notebook 3: Feature Engineering
## Translating Time-Series into Supervised Learning

**Objectives:**
1. Extract temporal features from dates (Year, Month, Day, DayOfWeek).
2. Create Lag features (`Sales_Lag_7`, `Sales_Lag_14`).
3. Compute Rolling Statistics (`RollingMean_7`).

*Constraint:* Native Python dictionary hashing is used for $O(1)$ lookups instead of `pandas.shift()`.


In [ ]:
from datetime import datetime, timedelta

def parse_date(date_str):
    return datetime.strptime(date_str, '%Y-%m-%d')

def create_temporal_features(row):
    dt = parse_date(row['Date'])
    row['Year'] = dt.year
    row['Month'] = dt.month
    row['Day'] = dt.day
    row['DayOfWeek'] = dt.weekday() + 1
    row['IsWeekend'] = 1 if row['DayOfWeek'] >= 6 else 0
    return row


### 1. Generating Lags via Dictionary Hashing


In [ ]:
# Build a hash map for O(1) lookups: key = (Store, DateStr)
sales_hash = {}
# Example population: sales_hash[(1, '2015-01-01')] = 5200.0

def get_lag_feature(store_id, current_date_str, lag_days, sales_hash):
    dt = parse_date(current_date_str)
    lag_dt = dt - timedelta(days=lag_days)
    lag_date_str = lag_dt.strftime('%Y-%m-%d')
    
    # O(1) lookup
    return sales_hash.get((store_id, lag_date_str), None)

print("Dictionary hashing ensures rapid feature extraction without Pandas DataFrames.")


### 2. Computing Rolling Means


In [ ]:
def get_rolling_mean(store_id, current_date_str, window_days, sales_hash):
    dt = parse_date(current_date_str)
    sales_window = []
    
    for i in range(1, window_days + 1):
        target_dt = dt - timedelta(days=i)
        val = sales_hash.get((store_id, target_dt.strftime('%Y-%m-%d')), None)
        if val is not None:
            sales_window.append(val)
            
    if not sales_window:
        return 0.0
    return sum(sales_window) / len(sales_window)

print("Engineered `RollingMean_7` and `RollingMean_30` to capture short-term and medium-term sales momentum.")
